# Exercício 2 · Duplicatas
**Ciência de Dados — Preparação dos Dados (CRISP-DM)**
**Aluno(a):** Geovanny Mahtus

**Base:** Customer Transaction Dataset (KPMG), arquivo `KPMG_VI_New_raw_data_update_final.xlsx`, aba *Transactions*.

**Contexto:** a empresa quer calcular faturamento, ticket médio e produtos mais vendidos. Se uma venda estiver contada duas vezes, todos esses números saem errados. O objetivo aqui é garantir que **cada transação apareça uma única vez**.

In [1]:
# IMPORTAÇÕES
import pandas as pd
import numpy as np

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = 20
pd.options.display.width = 150

In [2]:
# CARREGAMENTO DOS DADOS
arquivo = 'KPMG_VI_New_raw_data_update_final.xlsx'

print('Abas do arquivo:', pd.ExcelFile(arquivo).sheet_names)
leitura_direta = pd.read_excel(arquivo, sheet_name='Transactions')
leitura_direta.head(3)

Abas do arquivo: ['Title Sheet', 'Transactions', 'NewCustomerList', 'CustomerDemographic', 'CustomerAddress']


,Note: The data and information in this document is reflective of a hypothetical situation and client. This document is to be used for KPMG Virtual Internship purposes only.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
1,1,2,2950,2017-02-25 00:00:00,False,Approved,Solex,Standard,medium,medium,71.49,53.62,41245
2,2,3,3120,2017-05-21 00:00:00,True,Approved,Trek Bicycles,Standard,medium,large,"2,091.47",388.92,41701


A leitura direta **não funcionou**: os nomes das colunas viraram `Note: The data and information...` e `Unnamed: 1`, `Unnamed: 2`... A primeira linha da aba é um aviso da KPMG, e o cabeçalho de verdade está na linha de baixo.

Em vez de só trocar para `header=1`, escrevi uma função que **procura** a linha do cabeçalho. Assim eu descobri que **as abas não seguem o mesmo padrão**: em algumas o cabeçalho está na linha 0, em outras na linha 1.

In [3]:
# CONFERÊNCIA DO CABEÇALHO E DOS TIPOS
def ler_aba(arquivo, aba, coluna_chave='customer_id'):
    # olha as 10 primeiras linhas sem cabeçalho e usa como cabeçalho a primeira que contém a coluna-chave
    topo = pd.read_excel(arquivo, sheet_name=aba, header=None, nrows=10)
    linha = topo.index[topo.eq(coluna_chave).any(axis=1)][0]
    return pd.read_excel(arquivo, sheet_name=aba, header=linha), linha

for aba in ['Transactions', 'CustomerDemographic', 'CustomerAddress']:
    print(f'{aba:<20} -> cabeçalho na linha {ler_aba(arquivo, aba)[1]}')

df, _ = ler_aba(arquivo, 'Transactions')
print('\nDimensões:', df.shape)
pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isna().sum(), 'exemplo': df.iloc[0]})

Transactions         -> cabeçalho na linha 1


CustomerDemographic  -> cabeçalho na linha 0


CustomerAddress      -> cabeçalho na linha 1



Dimensões: (20000, 13)


,tipo,nulos,exemplo
transaction_id,int64,0,1
product_id,int64,0,2
customer_id,int64,0,2950
transaction_date,datetime64[us],0,2017-02-25 00:00:00
online_order,float64,360,0.00
order_status,str,0,Approved
brand,str,197,Solex
product_line,str,197,Standard
product_class,str,197,medium
product_size,str,197,medium


In [4]:
# Ajuste de tipos (em uma cópia; o df original continua igual)
trans = df.copy()

# product_first_sold_date está no formato serial do Excel (dias contados a partir de 30/12/1899)
trans['product_first_sold_date'] = pd.to_datetime(trans['product_first_sold_date'], unit='D', origin='1899-12-30')

# online_order é sim/não, mas veio como float por causa dos nulos -> booleano que aceita nulo
trans['online_order'] = trans['online_order'].map({1.0: True, 0.0: False}).astype('boolean')

print(trans[['product_first_sold_date', 'online_order']].dtypes)
print('Datas do 1º lançamento do produto:', trans['product_first_sold_date'].min().date(), 'a', trans['product_first_sold_date'].max().date())
print('Período das transações:', trans['transaction_date'].min().date(), 'a', trans['transaction_date'].max().date())
print(trans['order_status'].value_counts().to_dict())

product_first_sold_date    datetime64[s]
online_order                     boolean
dtype: object
Datas do 1º lançamento do produto: 1991-01-21 a 2016-12-06
Período das transações: 2017-01-01 a 2017-12-30
{'Approved': 19821, 'Cancelled': 179}


**Sobre os tipos**
- `product_first_sold_date` chegou como número (ex.: 41245). É uma data em formato serial do Excel; convertida, vai de 1991 a 2016, o que faz sentido para o lançamento de produtos.
- `online_order` chegou como `float` porque tem 360 nulos; virou booleano que aceita nulo.
- As colunas do produto (`brand`, `product_line`, `product_class`, `product_size`, `standard_cost`, `product_first_sold_date`) têm **os mesmos 197 nulos**: são transações em que o produto não foi cadastrado.
- Todas as transações são de 2017, e 179 estão como `Cancelled`.

In [5]:
# DUPLICATAS EXATAS
print('Linhas 100% iguais:', trans.duplicated().sum())
print('Linhas iguais se eu ignorar o transaction_id:', trans.drop(columns='transaction_id').duplicated().sum())

Linhas 100% iguais: 0
Linhas iguais se eu ignorar o transaction_id: 0


**Zero nos dois testes.** Isso é um bom sinal, mas não prova nada sozinho: se um registro foi lançado duas vezes com algum detalhe diferente (outro id, um campo em branco, um preço arredondado), ele escapa dessa checagem. Por isso o próximo passo é definir o que, para o negócio, significa "a mesma transação".

## Duplicatas por subconjunto de colunas
Testei uma "escada" de critérios, do mais frouxo ao mais rígido, para ver em que ponto as repetições deixam de aparecer:
1. **cliente + data**: o mesmo cliente comprou algo no mesmo dia;
2. **cliente + data + `product_id`**: o mesmo produto, segundo o código;
3. **cliente + data + produto descrito** (`brand`, `product_line`, `product_class`, `product_size`, `list_price`): o mesmo produto, segundo as características.

In [6]:
# DUPLICATAS POR SUBCONJUNTO DE COLUNAS
produto_descrito = ['brand', 'product_line', 'product_class', 'product_size', 'list_price']
criterios = {
    '1. cliente + data': ['customer_id', 'transaction_date'],
    '2. cliente + data + product_id': ['customer_id', 'transaction_date', 'product_id'],
    '3. cliente + data + produto descrito': ['customer_id', 'transaction_date'] + produto_descrito,
}
escada = pd.DataFrame({
    nome: {'linhas envolvidas': trans.duplicated(cols, keep=False).sum(),
           'cópias excedentes': trans.duplicated(cols).sum()}
    for nome, cols in criterios.items()
}).T
escada

,linhas envolvidas,cópias excedentes
1. cliente + data,264,132
2. cliente + data + product_id,2,1
3. cliente + data + produto descrito,0,0


In [7]:
# O que são as repetições do critério 1? Olho alguns exemplos
amostra = trans[trans.duplicated(criterios['1. cliente + data'], keep=False)]
print('Grupos cliente+data com mais de uma linha:', amostra.groupby(['customer_id', 'transaction_date']).ngroups)
print('Desses grupos, quantos têm marcas diferentes:',
      (amostra.groupby(['customer_id', 'transaction_date'])['brand'].nunique() > 1).sum())
display(amostra.sort_values(['customer_id', 'transaction_date'])
        [['transaction_id', 'customer_id', 'transaction_date', 'product_id', 'brand', 'list_price']].head(6))

print('Único caso do critério 2:')
trans[trans.duplicated(criterios['2. cliente + data + product_id'], keep=False)][
    ['transaction_id', 'customer_id', 'transaction_date', 'product_id', 'brand', 'product_size', 'list_price']]

Grupos cliente+data com mais de uma linha: 132
Desses grupos, quantos têm marcas diferentes: 115


,transaction_id,customer_id,transaction_date,product_id,brand,list_price
5984,5985,12,2017-08-21,39,Giant Bicycles,"1,812.75"
13817,13818,12,2017-08-21,44,WeareA2B,"1,769.64"
8349,8350,90,2017-03-09,99,OHM Cycles,"1,227.34"
17301,17302,90,2017-03-09,24,Solex,"1,777.80"
5541,5542,91,2017-07-28,11,Giant Bicycles,"1,274.93"
9662,9663,91,2017-07-28,7,Trek Bicycles,980.37


Único caso do critério 2:


,transaction_id,customer_id,transaction_date,product_id,brand,product_size,list_price
10910,10911,1840,2017-07-11,0,OHM Cycles,medium,235.63
16026,16027,1840,2017-07-11,0,Solex,large,202.62


**Escolha do critério**
- O critério 1 encontra **132 repetições**, mas são compras de **produtos diferentes** no mesmo dia: em 115 dos 132 grupos até a marca é diferente, e nos outros 17 muda o modelo ou o preço. Isso é um cliente levando duas bicicletas, não um erro. Usar esse critério apagaria vendas reais.
- O critério 2 encontra **um único par**, e ele também é falso: as duas linhas têm `product_id = 0`, mas marcas e preços diferentes. O problema aí é o código do produto, que não é confiável (ver próxima seção).
- **Escolhi o critério 3**: cliente + data + o produto descrito pelas suas características. Ele traduz a regra de negócio "o mesmo cliente não compra o mesmo produto, pelo mesmo preço, duas vezes no mesmo dia" sem depender do `product_id`. Por esse critério, a base real **não tem repetições**.
- Deixei de fora `transaction_id` (é gerado pelo sistema e muda num reenvio) e `online_order`/`order_status` (um reenvio pode chegar com esses campos em branco ou atualizados).

In [8]:
# UNICIDADE DOS IDENTIFICADORES
ids = trans['transaction_id']
print(f'transaction_id: {ids.nunique():,} valores para {len(trans):,} linhas | único? {ids.is_unique} | '
      f'sequência sem buracos? {ids.sort_values().diff().dropna().eq(1).all()}')

# product_id deveria apontar sempre para o mesmo produto
descricoes = trans.dropna(subset=['brand']).groupby('product_id')[produto_descrito].nunique()
print(f"\nproduct_id distintos: {trans['product_id'].nunique()} | produtos distintos pela descrição: "
      f"{trans[produto_descrito].dropna().drop_duplicates().shape[0]}")
print('product_id associados a mais de uma marca:', (descricoes['brand'] > 1).sum())
print('Pior caso:')
display(descricoes.sort_values('list_price', ascending=False).head(3))

# customer_id: os clientes das transações existem no cadastro?
demo, _ = ler_aba(arquivo, 'CustomerDemographic')
orfas = trans[~trans['customer_id'].isin(demo['customer_id'])]
print(f"customer_id único no cadastro? {demo['customer_id'].is_unique} | "
      f"transações de clientes que NÃO existem no cadastro: {len(orfas)} (clientes {orfas['customer_id'].unique().tolist()})")

transaction_id: 20,000 valores para 20,000 linhas | único? True | sequência sem buracos? True

product_id distintos: 101 | produtos distintos pela descrição: 100
product_id associados a mais de uma marca: 63
Pior caso:


,brand,product_line,product_class,product_size,list_price
product_id,,,,,
0,6,2,3,3,25
1,1,2,1,2,2
2,2,2,2,2,2


customer_id único no cadastro? True | transações de clientes que NÃO existem no cadastro: 3 (clientes [5034])


**O que descobri sobre os identificadores**
- `transaction_id` é **confiável**: 20.000 valores diferentes, de 1 a 20.000, sem buracos.
- `product_id` **não identifica um produto**: existem 101 códigos e 100 produtos diferentes pela descrição (marca, linha, classe, tamanho e preço), números parecidos que **não se correspondem**: 63 códigos aparecem com mais de uma marca. O código **0** é o pior: 6 marcas e 25 preços diferentes, ou seja, funciona como um "código genérico". Qualquer ranking de "produto mais vendido" feito por `product_id` sairia errado.
- `customer_id` é único no cadastro, mas **3 transações** pertencem ao cliente **5034**, que não existe na aba de clientes. Não é duplicata, é uma quebra de integridade entre as tabelas, e precisa ser reportada.

## Estratégia de tratamento
**Na base real** o critério escolhido não encontrou repetições, então **nenhuma linha é removida**. Em vez de parar aqui, deixei a regra pronta e testei com o tipo de duplicata mais comum na prática: uma **carga dobrada**. Simulei que o sistema reimportou as vendas dos últimos 3 dias de dezembro com ids novos e, em parte delas, com `online_order` em branco (como acontece quando uma importação é parcial).

**Estratégia:** para cada grupo do critério 3, **consolidar** as linhas: manter o `transaction_id` mais antigo (o primeiro lançamento) e, se algum campo estiver vazio nele, aproveitar o valor da cópia. Assim nenhuma informação é perdida e a venda passa a ser contada uma vez só.

In [9]:
# ESTRATÉGIA DE TRATAMENTO E IMPACTO
chave = criterios['3. cliente + data + produto descrito']

def deduplicar(base, chave):
    # ordena pelo id e junta cada grupo: first() pega o primeiro valor NÃO nulo de cada coluna
    ordenada = base.sort_values('transaction_id')
    consolidada = ordenada.groupby(chave, dropna=False, sort=False).first().reset_index()
    return consolidada[base.columns]

def indicadores(base):
    aprovadas = base[base['order_status'] == 'Approved']
    return pd.Series({
        'transações': len(base),
        'faturamento (soma list_price)': aprovadas['list_price'].sum(),
        'ticket médio': aprovadas['list_price'].mean(),
        'lucro bruto (list_price - standard_cost)': (aprovadas['list_price'] - aprovadas['standard_cost']).sum(),
    })

# 1) base real
trans_limpa = deduplicar(trans, chave)
print('Base real -> linhas afetadas:', len(trans) - len(trans_limpa))

# 2) simulação de carga dobrada
ultimos_dias = trans[trans['transaction_date'] >= '2017-12-28'].copy()
ultimos_dias['transaction_id'] = np.arange(len(ultimos_dias)) + trans['transaction_id'].max() + 1
ultimos_dias.loc[ultimos_dias.sample(frac=0.4, random_state=11).index, 'online_order'] = pd.NA
com_carga_dobrada = pd.concat([trans, ultimos_dias], ignore_index=True)
corrigida = deduplicar(com_carga_dobrada, chave)

print('Simulação -> linhas reinseridas:', len(ultimos_dias))
print('   duplicatas exatas detectadas:', com_carga_dobrada.duplicated().sum())
print('   duplicatas pelo critério 3  :', com_carga_dobrada.duplicated(chave).sum())
print('   linhas removidas/consolidadas:', len(com_carga_dobrada) - len(corrigida))

impacto = pd.DataFrame({
    'base real': indicadores(trans),
    'base real após a regra': indicadores(trans_limpa),
    'com carga dobrada': indicadores(com_carga_dobrada),
    'carga dobrada corrigida': indicadores(corrigida),
})
impacto['erro se não tratar (%)'] = (impacto['com carga dobrada'] / impacto['base real'] - 1) * 100
impacto

Base real -> linhas afetadas: 0
Simulação -> linhas reinseridas: 157
   duplicatas exatas detectadas: 0
   duplicatas pelo critério 3  : 157
   linhas removidas/consolidadas: 157


,base real,base real após a regra,com carga dobrada,carga dobrada corrigida,erro se não tratar (%)
transações,"20,000.00","20,000.00","20,157.00","20,000.00",0.78
faturamento (soma list_price),"21,954,196.69","21,954,196.69","22,131,468.08","21,954,196.69",0.81
ticket médio,"1,107.62","1,107.62","1,107.85","1,107.62",0.02
lucro bruto (list_price - standard_cost),"10,831,504.03","10,831,504.03","10,916,536.32","10,831,504.03",0.79


In [10]:
# Conferência: a consolidação devolveu exatamente a base original?
volta = corrigida.sort_values('transaction_id').reset_index(drop=True)
print('Mesmo conjunto de transaction_id da base real?', set(volta['transaction_id']) == set(trans['transaction_id']))
print('online_order nulos — original:', trans['online_order'].isna().sum(), '| após consolidar:', volta['online_order'].isna().sum())

Mesmo conjunto de transaction_id da base real? True
online_order nulos — original: 360 | após consolidar: 360


**Impacto**
- **Base real:** 0 registros afetados, então os indicadores ficam idênticos. A decisão foi **não remover nada** e manter a regra documentada.
- **Carga dobrada:** os 157 registros reimportados (0,8% da base) passam **despercebidos** pela checagem de duplicatas exatas (resultado 0), mas o critério 3 encontra todos. Sem tratamento, o faturamento ficaria **inflado em cerca de 0,8%** e o número de transações também. Depois da consolidação, os indicadores voltam a ser iguais aos da base real e o conjunto de `transaction_id` é exatamente o original.
- **Consolidar em vez de só apagar** evitou perder informação: como o lançamento original é mantido e os campos vazios são completados pela cópia, a quantidade de `online_order` nulos não aumenta.

## Desafio 1 — Textos parecidos com `rapidfuzz`
Uso `process.cdist`, que compara todos os textos contra todos de uma vez, e listo os pares com similaridade acima de um corte.

In [11]:
from rapidfuzz import fuzz, process, utils
import re

def pares_parecidos(serie, corte=90, scorer=fuzz.token_sort_ratio):
    valores = serie.dropna().unique()
    matriz = process.cdist(valores, valores, scorer=scorer, processor=utils.default_process, score_cutoff=corte)
    i, j = np.where(np.triu(matriz, k=1) > 0)
    return (pd.DataFrame({'texto_1': valores[i], 'texto_2': valores[j], 'similaridade': matriz[i, j]})
              .sort_values('similaridade', ascending=False, ignore_index=True))

print('brand:', trans['brand'].dropna().unique().tolist())
print('Pares parecidos em brand:', len(pares_parecidos(trans['brand'])))

pares_cargo = pares_parecidos(demo['job_title'])
print(f"\njob_title: {demo['job_title'].nunique()} cargos distintos, {len(pares_cargo)} pares com similaridade >= 90")
pares_cargo.head(8)

brand: ['Solex', 'Trek Bicycles', 'OHM Cycles', 'Norco Bicycles', 'Giant Bicycles', 'WeareA2B']
Pares parecidos em brand: 0

job_title: 195 cargos distintos, 161 pares com similaridade >= 90


,texto_1,texto_2,similaridade
0,Budget/Accounting Analyst II,Budget/Accounting Analyst III,98.25
1,Human Resources Assistant II,Human Resources Assistant III,98.25
2,Human Resources Assistant I,Human Resources Assistant II,98.18
3,Budget/Accounting Analyst I,Budget/Accounting Analyst IV,98.18
4,Administrative Assistant III,Administrative Assistant II,98.18
5,Human Resources Assistant I,Human Resources Assistant IV,98.18
6,Budget/Accounting Analyst I,Budget/Accounting Analyst II,98.18
7,Computer Systems Analyst II,Computer Systems Analyst III,98.18


In [12]:
# Os pares de job_title são o mesmo cargo escrito diferente ou níveis diferentes?
sem_nivel = demo['job_title'].dropna().str.replace(r'\s+(I|II|III|IV|V|VI|VII|VIII|IX|X)$', '', regex=True)
print('Cargos distintos com o nível (I, II, III...):', demo['job_title'].nunique())
print('Cargos distintos sem o nível:               ', sem_nivel.nunique())
print('Pares parecidos depois de tirar o nível:     ', len(pares_parecidos(sem_nivel)))

# E a demonstração do enunciado: o caso "Giant Bicycles" x "Giant Bicycle"
print('\n"Giant Bicycles" x "Giant Bicycle":', round(fuzz.token_sort_ratio('Giant Bicycles', 'Giant Bicycle'), 1))

Cargos distintos com o nível (I, II, III...): 195
Cargos distintos sem o nível:                114
Pares parecidos depois de tirar o nível:      1

"Giant Bicycles" x "Giant Bicycle": 96.3


In [13]:
# Um caso real em que o fuzzy NÃO resolve: a coluna state da aba CustomerAddress
endereco, _ = ler_aba(arquivo, 'CustomerAddress')
print(endereco['state'].value_counts().to_string())

for a, b in [('NSW', 'New South Wales'), ('VIC', 'Victoria')]:
    print(f'{a!r} x {b!r}: token_sort_ratio = {fuzz.token_sort_ratio(a, b):.0f} | WRatio = {fuzz.WRatio(a, b, processor=utils.default_process):.0f}')

siglas = {'New South Wales': 'NSW', 'Victoria': 'VIC', 'Queensland': 'QLD'}
estado_padronizado = endereco['state'].replace(siglas)
print('\nDepois do dicionário de siglas:', estado_padronizado.value_counts().to_dict())

state
NSW                2054
VIC                 939
QLD                 838
New South Wales      86
Victoria             82
'NSW' x 'New South Wales': token_sort_ratio = 33 | WRatio = 60
'VIC' x 'Victoria': token_sort_ratio = 18 | WRatio = 90

Depois do dicionário de siglas: {'NSW': 2140, 'VIC': 1021, 'QLD': 838}


**O que o desafio mostrou**
- Em `brand` **não há** variações de escrita: a coluna já está limpa. Mesmo assim, o rapidfuzz dá 96 de similaridade para "Giant Bicycles" x "Giant Bicycle", então pegaria esse erro se ele existisse.
- Em `job_title` aparecem **mais de 160 pares** acima de 90, mas quase todos são **níveis diferentes do mesmo cargo** ("Analyst I" x "Analyst II"). São falsos positivos: o texto é quase igual, o significado não. Tirando o numeral romano do fim, os 195 cargos viram 114 e sobra só 1 par parecido. Lição: o fuzzy **sugere candidatos**; quem decide é uma pessoa olhando os casos.
- Em `state`, o mesmo estado aparece como sigla e por extenso (`NSW` x `New South Wales`, `VIC` x `Victoria`). Aqui o fuzzy **não é confiável**: o `token_sort_ratio` dá só 33 e 18. O `WRatio`, que compara trechos, chega a 90 em `VIC` x `Victoria` (porque a sigla coincide com o começo da palavra), mas fica em 60 em `NSW` x `New South Wales`. Funcionar ou não depende do acaso das letras. O jeito certo foi um **dicionário de padronização**: o fuzzy pega erros de digitação, não abreviações.

## Desafio 2 — Gênero na aba CustomerDemographic

In [14]:
print('Valores como estão no arquivo:')
print(demo['gender'].value_counts(dropna=False).to_string())

padronizado = demo['gender'].str.strip().str.upper().str[0]          # primeira letra: F, M, U
print('\nAgrupando pela primeira letra:', padronizado.value_counts(dropna=False).to_dict())
print('Formas diferentes de registrar gênero:', demo['gender'].nunique(dropna=False))

Valores como estão no arquivo:
gender
Female    2039
Male      1873
U           88

Agrupando pela primeira letra: {'F': 2039, 'M': 1873, 'U': 88}
Formas diferentes de registrar gênero: 3


**Resultado:** nesta versão do arquivo existem **3 formas**: `Female` (2.039), `Male` (1.873) e `U` (88). Não há variações como `F`, `Femal` ou `M` (que existem em outras versões desse dataset). A função acima, que agrupa pela primeira letra, já trataria esses casos se aparecessem.

O ponto de atenção é o **`U`**: significa *Unknown*, ou seja, gênero **não informado**. Ele não deveria ser tratado como uma terceira categoria em um gráfico de "vendas por gênero"; o correto é convertê-lo em ausente (`NaN`) ou deixá-lo explicitamente como "não informado".

## Respostas às perguntas de reflexão

**1. Se o resultado de duplicatas exatas for zero, isso garante que não há registros repetidos? Por quê?**
Não garante. O `duplicated()` só acusa linhas iguais em **todas** as colunas. Basta uma diferença, mesmo irrelevante, para a cópia passar: um id novo gerado na reimportação, um campo que veio em branco, um espaço no fim do texto. Na simulação da carga dobrada, a checagem exata deu **0** enquanto havia **157** vendas contadas duas vezes. Por isso é preciso testar também por subconjunto de colunas e, em textos, por similaridade.

**2. Em uma base de transações, duas linhas idênticas são sempre um erro? Dê um exemplo em que não seriam.**
Não. Se a base não tem coluna de quantidade nem de horário, uma pessoa que compra **dois capacetes iguais** e passa cada um separadamente no caixa gera duas linhas idênticas, e as duas são vendas reais. Outro caso é uma mensalidade de academia: o mesmo valor, para o mesmo cliente, lançado todo mês, fica igual se a base só guarda o mês de referência em outra tabela. Quando o nível de detalhe da base não diferencia os eventos, linha igual não significa erro.

**3. Por que a escolha das colunas que definem "o mesmo registro" é uma decisão de negócio? Relacione com o Entendimento do Negócio do CRISP-DM.**
Porque a pergunta "isto é a mesma venda?" depende de como a empresa funciona, e não do código. Neste exercício, o critério cliente + data apagaria 132 compras legítimas de quem levou mais de uma bicicleta no mesmo dia; o critério com `product_id` confiaria num código que, como vimos, não identifica o produto. Só quem conhece o negócio sabe responder se um cliente pode comprar o mesmo item duas vezes no dia, se o preço pode mudar entre um lançamento e outro e qual é o nível de detalhe de cada linha (item, pedido, nota fiscal). No CRISP-DM, essas respostas vêm da etapa de **Entendimento do Negócio**, que define o que será medido e com que granularidade. A etapa de Preparação dos Dados apenas transforma essa definição em código; se a definição estiver errada, a limpeza vai remover vendas reais ou deixar passar repetições.